# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muneebulhaq02/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muneebulhaq02/flyrank-ml-internship"
REPO_DIR = "FlyRank-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/FlyRank-Internship


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The objective of this step is to build a feature vector that can be used to prioritize content pages for review. I selected features that are available before the prediction moment and describe search visibility, user engagement, and content characteristics. Label-derived fields, future information, and product-generated flags are deliberately excluded to reduce leakage. Missing values are handled using simple imputations or indicator variables where appropriate so that the resulting feature matrix is complete for modeling.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "content_age_days",
    "search_volume",
    "word_count",
    "engagement_rate"
]

X = df[features].copy()

X["word_count"] = X["word_count"].fillna(
    X["word_count"].median()
)

X["engagement_rate"] = X["engagement_rate"].fillna(
    X["engagement_rate"].median()
)

print("Feature Matrix Shape:", X.shape)
print()
print(X.head())

Feature Matrix Shape: (30000, 7)

   impressions_90d  clicks_90d   ctr  content_age_days  search_volume  \
0             3803          29  0.76               187           10.0   
1            15320           7  0.05               445           90.0   
2            12581          11  0.09               141            0.0   
3            11751          58  0.49               463           10.0   
4            19140          24  0.13               263            0.0   

   word_count  engagement_rate  
0      3221.0             5.88  
1      2481.0             0.00  
2      3515.0             0.00  
3      2877.0             1.28  
4      2803.0             0.00  


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

2. Feature notes (meaning, missing, categorical, available-when?)

The selected features represent information that is available before making a prioritization decision. Impressions, clicks, CTR, search volume, and engagement describe historical performance, while content age and word count describe page characteristics. Missing values in word count and engagement rate are replaced with the median because these fields contain incomplete observations. None of the selected features depend on future outcomes or label-derived calculations, making them appropriate inputs for prediction.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_notes = pd.DataFrame({

    "Feature":[
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "content_age_days",
        "search_volume",
        "word_count",
        "engagement_rate"
    ],

    "Missing Values":[
        df["impressions_90d"].isna().sum(),
        df["clicks_90d"].isna().sum(),
        df["ctr"].isna().sum(),
        df["content_age_days"].isna().sum(),
        df["search_volume"].isna().sum(),
        df["word_count"].isna().sum(),
        df["engagement_rate"].isna().sum()
    ],

    "Available Before Prediction":[
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]

})

print(feature_notes)

            Feature  Missing Values Available Before Prediction
0   impressions_90d               0                         Yes
1        clicks_90d               0                         Yes
2               ctr               0                         Yes
3  content_age_days               0                         Yes
4     search_volume            2468                         Yes
5        word_count            7699                         Yes
6   engagement_rate               0                         Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

A careful leakage review is necessary before training any model. According to the FlyRank dataset documentation, trend_direction, trend_pct, and is_declining_label are directly related to the target and therefore must never be used as features. Likewise, pseudonymous identifiers such as content_id and client_id are used only for grouping or validation and not as predictive variables. This notebook checks that these columns are not present in the feature matrix before modeling.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leaky_columns = [

    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"

]

print("Leakage Check")
print("-"*40)

for column in leaky_columns:

    if column in X.columns:
        print(column,"FOUND in features")
    else:
        print(column,"Not used")

print()
print("Total Features Used:",len(X.columns))
print("Feature Names:")
print(list(X.columns))

Leakage Check
----------------------------------------
trend_direction Not used
trend_pct Not used
is_declining_label Not used
content_id Not used
client_id Not used

Total Features Used: 7
Feature Names:
['impressions_90d', 'clicks_90d', 'ctr', 'content_age_days', 'search_volume', 'word_count', 'engagement_rate']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Several fields were intentionally excluded from the feature vector. Trend-related columns were removed because they contribute directly to the target label and would introduce leakage. Pseudonymous identifiers such as client_id and content_id were excluded because they uniquely identify entities rather than describe page behavior. Existing product decisions or future-derived information were also avoided because the objective is to learn from observable page characteristics rather than reproduce previous decisions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).

excluded = pd.DataFrame({

    "Excluded Column":[
        "trend_direction",
        "trend_pct",
        "is_declining_label",
        "client_id",
        "content_id"
    ],

    "Reason":[
        "Derived from target",
        "Label-related feature",
        "Prediction target",
        "Identifier only",
        "Identifier only"
    ]

})

print(excluded)

      Excluded Column                 Reason
0     trend_direction    Derived from target
1           trend_pct  Label-related feature
2  is_declining_label      Prediction target
3           client_id        Identifier only
4          content_id        Identifier only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.